# DataPulse AI — Silent ML Data Failure, From Scratch

**Goal:** demonstrate that conventional data quality can pass while a hidden semantic drift changes ML decisions. All data are synthetic. This runs on free Google Colab CPU and does not require GCP credentials for the core demo.

**Experiment:** fixed churn model + identical test cohort + trusted vs changed feature snapshots.


## 1. Download the working project from GitHub

Run this code cell. It clones the public repository into the temporary Colab runtime and enters the standalone `datapulse-ai` folder.


In [ ]:
from pathlib import Path
import subprocess, os
repo = Path('/content/aegis-real-time-supply-chain-ai')
if not (repo / '.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Codewithadi21/aegis-real-time-supply-chain-ai.git',str(repo)],check=True)
os.chdir(repo / 'datapulse-ai')
print('Working directory:',os.getcwd())


## 2. Install dependencies and run automated tests

In [ ]:
%pip -q install -r requirements.txt
!python -m pytest -q


## 3. Generate trusted baseline data and an intentionally semantically broken snapshot

In [ ]:
from datapulse.generator import make_customer_data, introduce_semantic_shift, introduce_formatting_faults
from datapulse.quality import validate_contract, clean_formatting
baseline = make_customer_data(5000)
shifted = introduce_semantic_shift(baseline)
print('Trusted snapshot:'); display(baseline.head())
print('Shifted snapshot:'); display(shifted.head())
print('Basic checks, both snapshots:')
print(validate_contract(baseline))
print(validate_contract(shifted))


## 4. Investigate why traditional checks are insufficient

The values are all valid integers, no missing columns or duplicates, yet `engagement_events_30d` has silently changed from **purchases** to **logins**. The semantic contract can compare an independently trusted purchase counter in this controlled experiment.


In [ ]:
from datapulse.drift import semantic_contract,feature_checks
print('Semantic check baseline:',semantic_contract(baseline))
print('Semantic check shifted:',semantic_contract(shifted))
print('Statistical drift:',feature_checks(baseline,shifted,'engagement_events_30d'))


## 5. Show the simpler whitespace / duplicates issue too

In [ ]:
broken_formatting = introduce_formatting_faults(baseline)
print('Detected formatting source errors:',validate_contract(broken_formatting))
cleaned = clean_formatting(broken_formatting)
print('After normalization and key dedup:',validate_contract(cleaned))


## 6. Model impact replay — same model, same customers

We train on an unchanged training cohort, then score a fixed holdout cohort twice. We compare both probability and binary decision changes. Labels are synthetic and known here; in real production they may arrive later.


In [ ]:
from datapulse.replay import decision_replay
from datapulse.investigator import investigate
summary, comparisons = decision_replay()
for key in ['test_rows','prediction_flips','prediction_flip_rate','baseline_model','changed_data_same_model']:
    print(key,summary[key])
display(comparisons[comparisons.decision_changed].head(12))


## 7. Visualize changed decisions

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
values=[summary['baseline_model']['false_negatives'],summary['changed_data_same_model']['false_negatives']]
ax.bar(['Trusted snapshot','Changed snapshot'],values)
ax.set_title('Missed actual churners (synthetic labeled cohort)')
ax.set_ylabel('False-negative count')
plt.show()


## 8. Generate an explainable, evidence-based incident report

In [ ]:
import json
report=investigate(summary)
print(json.dumps(report,indent=2))
!python -m datapulse --rows 5000


## 9. Save outputs and capture evidence for GitHub

The CLI saves JSON reports and a paired prediction CSV in `artifacts/`. These outputs are gitignored because they are run artifacts; keep screenshots or attach reproducible examples if desired.


In [ ]:
from pathlib import Path
print('
'.join(str(p) for p in Path('artifacts').glob('*')))


## 10. Optional: Connect your personal GCP BigQuery account

Only do this if you want cloud evidence and have confirmed your GCP trial credits. **No cloud account is required above.** The next cells should be run manually with your own project ID; this is not automatic deployment.


In [ ]:
# Uncomment only when ready to use your personal GCP project:
# from google.colab import auth
# auth.authenticate_user()
# PROJECT_ID = 'YOUR_ACTUAL_GCP_PROJECT_ID'
# %pip -q install -r requirements-gcp.txt
# !python -m cloud.bigquery_load --project-id {PROJECT_ID}


## 11. Optional: Publish a synthetic incident to Pub/Sub

This only publishes a message. It does not create a live subscriber or an always-on streaming service.


In [ ]:
# Uncomment after enabling Pub/Sub API and setting PROJECT_ID above:
# !python -m cloud.pubsub_demo --project-id {PROJECT_ID}


## 12. Optional: Gemini explanation of verified evidence

Do **not** paste your API key in a notebook that you commit. Use a Colab secret or private environment variable; API usage may incur charges. This step is optional and does not affect the measured metrics.


In [ ]:
# Only if GOOGLE_API_KEY is privately configured:
# from datapulse.investigator import optional_gemini_explanation
# print(optional_gemini_explanation(report, model_name='gemini-3.8-flash'))


## 13. Engineering questions you should be able to answer

- What is the difference between a parse/type failure and a semantic definition change?
- Why does a paired replay use a fixed model and customer cohort?
- When are PSI and KS meaningful, and when are they misleading?
- Why does a prediction flip not automatically prove business damage?
- How would you monitor feature definitions across teams and enforce producer contracts?
- How would you make the incident investigation agent read-only, deterministic and auditable?

Read `docs/INTERVIEW_GUIDE.md` for more.
